# Portfolio Builder: Signals → Optimization → Execution

**Learning objectives:**
- Build multi-asset portfolios from signals
- Apply different optimization methods
- Analyze portfolio performance
- Compare strategies (momentum, risk parity, blended)

This notebook demonstrates the complete research-to-portfolio workflow.

## 1. Setup

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

project_root = Path('/Users/zelin/Desktop/PA Investment/Invest_strategy')
sys.path.insert(0, str(project_root))

from portfolio.builder import PortfolioBuilder, PortfolioConfig, MultiAssetStrategy
from portfolio.optimizer import mean_variance_optimize, OptimizationConfig
from portfolio.risk import ledoit_wolf_cov

print("Setup complete")

## 2. Define Universe

In [ ]:
# Multi-asset universe
universe = {
    'SPY': 'S&P 500 ETF',
    'TLT': '20+ Year Treasury',
    'GLD': 'Gold ETF',
    'UUP': 'US Dollar ETF',
    'VNQ': 'Real Estate ETF',
}

start_date = '2015-01-01'
end_date = '2024-12-31'

# Data loader function
def data_loader(ticker, start, end):
    data = yf.download(ticker, start=start, end=end, progress=False)
    return data

print(f"Universe: {list(universe.keys())}")
print(f"Period: {start_date} to {end_date}")

## 3. Build Portfolio with Signals

In [ ]:
# Create portfolio builder
config = PortfolioConfig(
    universe=list(universe.keys()),
    signals=['momentum_60_21', 'mean_reversion'],
    optimization='mean_variance',
    risk_aversion=1.5,
    max_weight=0.30,
    min_weight=-0.10,
)

builder = PortfolioBuilder(config)
builder.load_data(data_loader, start_date, end_date)

# Compute signals
builder.compute_signals()

# Generate alpha
builder.generate_alpha(method='mean')

# Optimize weights
builder.optimize_weights()

# Summary
builder.summary()

## 4. Compare Optimization Methods

In [ ]:
# Compare different optimization methods
methods = ['equal_weight', 'risk_parity', 'mean_variance']

results = {}
for method in methods:
    config = PortfolioConfig(
        universe=list(universe.keys()),
        signals=['momentum_60_21'],
        optimization=method,
        risk_aversion=1.5,
    )
    
    builder = PortfolioBuilder(config)
    builder.load_data(data_loader, start_date, end_date)
    builder.compute_signals()
    builder.optimize_weights()
    
    weights = builder.weights
    results[method] = weights
    
    print(f"
{method.upper()}:")
    print(weights.sort_values(ascending=False).to_string())

# Plot comparison
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(universe))
width = 0.25

for i, (method, weights) in enumerate(results.items()):
    ax.bar(x + i*width, [weights.get(k, 0) for k in universe.keys()], width, label=method)

ax.set_ylabel('Weight')
ax.set_title('Portfolio Weights by Optimization Method')
ax.set_xticks(x + width)
ax.set_xticklabels(universe.keys())
ax.legend()
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 5. Risk Parity Strategy

In [ ]:
# Pure risk parity (no signals)
builder_rp = MultiAssetStrategy.risk_parity_strategy(
    universe=list(universe.keys()),
    start=start_date,
    end=end_date,
)

builder_rp.summary()

## 6. Trend Following Strategy

In [ ]:
# Trend following with momentum signal
builder_tf = MultiAssetStrategy.trend_following(
    universe=list(universe.keys()),
    start=start_date,
    end=end_date,
)

builder_tf.summary()

## 7. Blended Strategy

In [ ]:
# Blended: momentum + mean reversion
builder_blend = MultiAssetStrategy.blended_strategy(
    universe=list(universe.keys()),
    start=start_date,
    end=end_date,
)

builder_blend.summary()

## 8. Custom Optimization

In [ ]:
# Load data
config = PortfolioConfig(universe=list(universe.keys()))
builder = PortfolioBuilder(config)
builder.load_data(data_loader, start_date, end_date)

# Get returns
returns = builder.prices.pct_change().dropna()

# Compute covariance
cov = ledoit_wolf_cov(returns)

# Custom expected returns (e.g., equal weight momentum)
from research.signals import MomentumSignal
momentum = MomentumSignal(lookback=60, skip=21)
mom = momentum.compute(builder.prices[['close']])

# Normalize to expected returns
alpha = (mom - mom.mean()) / mom.std()
expected_returns = alpha * returns.std() * np.sqrt(252)

# Custom optimization
cfg = OptimizationConfig(
    risk_aversion=2.0,
    max_weight=0.25,
    min_weight=-0.10,
    turnover_aversion=0.1,  # Add turnover penalty
)

weights = mean_variance_optimize(
    expected_returns=expected_returns,
    cov=cov,
    cfg=cfg,
)

print("Custom Optimized Weights:")
print(weights.sort_values(ascending=False).to_string())

## 9. Summary

In [ ]:
print("""
=== KEY TAKEAWAYS ===

1. **Portfolio Builder Flow**
   - Load data → Compute signals → Generate alpha → Optimize weights
   - All-in-one pipeline for multi-asset strategies

2. **Optimization Methods**
   - Equal Weight: Simple 1/N allocation
   - Risk Parity: Equal risk contribution
   - Mean Variance: Markowitz with signal-derived alpha
   - Custom: Full control over parameters

3. **Key Parameters**
   - risk_aversion: Higher = more conservative
   - max_weight: Single asset cap
   - turnover_penalty: Reduce trading frequency
   - min_weight: Allow shorting

4. **Next Steps**
   - Add walk-forward validation to portfolios
   - Use more sophisticated signals
   - Connect to execution for paper/live trading
""")